In [1]:
%load_ext autoreload
%autoreload 2

import mangrove


In [ ]:

# instantiate imaging object with default hydra params
imaging_object = mangrove.acquire.forest_acquisition(default.yaml)

# update acquisition parameters (leaves the device intact, but updates configurations)
imaging_object.update_params(default.yaml,tx_cycles=4)

# acquire data from the probe or from previously acquired data
data_obj = imaging_object.acquire() #  from the probe
data_obj = mangrove.io.load_dataobj(/path/to/data/folder) # from previous data


# process the data using hydra config
data_obj.beamform(default_beamforming_params.yaml)
data_obj.clutter_filter(default_clutter_params.yaml) # name for algo that goes from ensembles to power doppler

# process the data, with different params
data_obj.beamform(default_beamforming_params.yaml,beamform_spacing=4)
data_obj.select_clutter_mask()
data_obj.clutter_filter(default_clutter_params.yaml, filter_params=[50 -1])

# Note that we may use different beamforming approaches on the same data. This will use different code, not just different params. 
# Likewise for clutter filtering. Further, each may invlve a series of preprocessing/post processing stages. So e.g. "beamform" should
# ultimetly point to a modular processing pipline that can be updated with relatively low overhead

# plot various things
data_obj.plot_spectrogram() # plots 
data_obj.plot_UPUsequence() # plots 
data_obj.plot_bMode(dbscale=40) # plots 

In [ ]:
# allow grabbing info out for rapid prototyping
bMode_data,bMode_meta_data=data_obj.get_bmode_data

from characterization_repo import estimate_CNR
estimate_CNR(bMode_data,bMode_meta_data)

In [ ]:
# run a simple experiment by iterating over parameters with a for-loop
data_objs = []  # Initialize an empty list to store data_obj
tx_cycles=[2,3,4,5]
tx_freq_hz=[5e6,6e6,7e6]

# Using a for-loop to iterate over both parameters
for index, (cycles, freq_hz) in enumerate(zip(tx_cycles, tx_freq_hz)):
    imaging_object.update_params(default, tx_cycles=cycles, tx_freq_hz=freq_hz)
    data_obj = imaging_object.acquire()
    data_obj.beamform(default)
    data_obj.clutter_filter(default)

    data_objs.append(data_obj)


# alternatively we do not need to collect data, but can instead just spit out config files
for index, (cycles, freq_hz) in enumerate(zip(tx_cycles, tx_freq_hz)):
    imaging_object.update_params(default, tx_cycles=cycles, tx_freq_hz=freq_hz)
    imaging_object.saveconfig(/path/to/save/dir)

    data_objs.append(data_obj)



data_objs was a simple list, but we may consider a utility class that enables helpful functions over a set of data_objs, e.g. it would enable something like what is below, plottign all X (e.g. bmodes) over the set of objects it contains.

In [ ]:


import matplotlib.pyplot as plt

# Determine the number of subplots needed based on the length of beam_formeds list
num_plots = len(data_objs)
cols = 2  # Define the number of columns in the subplot grid
rows = num_plots // cols + (num_plots % cols > 0)  # Calculate rows needed

fig, axs = plt.subplots(rows, cols, figsize=(10, rows*5))  # Initialize subplot with dynamic sizing based on rows
fig.suptitle('BMode Images')

# Flatten the axs array for easy indexing if there's more than one row
if rows > 1:
    axs = axs.flatten()

for index, beam_formed in enumerate(beam_formeds):
    ax = axs[index] if num_plots > 1 else axs  # Select the appropriate subplot
    data_objs.plot_bMode(ax=ax)  # Plot the bMode image on the selected subplot
    ax.set_title(f'Cycle: {tx_cycles[index]}, Freq: {tx_freq_hz[index]/1e6}MHz')  # Set title with parameters

# Hide any unused subplots
if num_plots > 1:
    for index in range(num_plots, len(axs)):
        fig.delaxes(axs[index])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to make room for the main title
plt.show()